Install the libraries

In [1]:
%pip install -U pandas numpy matplotlib scikit-learn joblib \
    sentence-transformers transformers datasets accelerate evaluate torch


  Using cached matplotlib-3.10.9-cp310-cp310-win_amd64.whl (8.2 MB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
  Using cached sentence_transformers-5.6.1-py3-none-any.whl (596 kB)
  Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
  Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
  Using cached torch-2.13.0-cp310-cp310-win_amd64.whl (122.0 MB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
  Using cached huggingface_hub-1.25.1-py3-none-any.whl (774 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
  Using cached typer-0.27.0-py3-none-any.whl (122 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached multiprocess-0.70.19-py310-none-any.whl (134 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached aiohttp-3.14.3-cp310-cp310-win_amd64.whl (480 kB)
  Using ca

ERROR: Could not install packages due to an OSError: [WinError 206] The filename or extension is too long: 'C:\\Users\\Admin\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python310\\site-packages\\torch-2.13.0.dist-info\\licenses\\third_party\\flash-attention\\third_party\\aiter\\3rdparty\\composable_kernel\\docs'


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


NOTEBOOK ONE. DATA PREPARATION.

Import and paths

In [2]:
from pathlib import Path
import ast
import random
import re

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "processed_data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"

for directory in [
    PROCESSED_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    PREDICTIONS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

for path in [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

print("All dataset files were found.")

All dataset files were found.


load the data

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
validation_raw = pd.read_csv(VALIDATION_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("Train:", train_raw.shape)
print("Validation:", validation_raw.shape)
print("Test:", test_raw.shape)

display(train_raw.head())

Train: (11118, 3)
Validation: (1000, 3)
Test: (1000, 3)


,dialog,act,emotion
0,"['Say , Jim , how about going for a few beers ...",[3 4 2 2 2 3 4 1 3 4],[0 0 0 0 0 0 4 4 4 4]
1,"['Can you do push-ups ? '\n "" Of course I can ...",[2 1 2 2 1 1],[0 0 6 0 0 0]
2,"['Can you study with the radio on ? '\n ' No ,...",[2 1 2 1 1],[0 0 0 0 0]
3,['Are you all right ? '\n ' I will be all righ...,[2 1 1 1],[0 0 0 0]
4,"['Hey John , nice skates . Are they new ? '\n ...",[2 1 2 1 1 2 1 3 4],[0 0 0 0 0 6 0 6 0]


validating the columns

In [4]:
REQUIRED_COLUMNS = {"dialog", "act", "emotion"}

for split_name, dataframe in {
    "train": train_raw,
    "validation": validation_raw,
    "test": test_raw
}.items():

    missing_columns = REQUIRED_COLUMNS - set(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"{split_name} is missing columns: {missing_columns}"
        )

print("All required columns are present.")

All required columns are present.


Correct parsing functions.

The dialogue strings are NumPy-style arrays containing adjacent quoted strings.

In [5]:
QUOTED_STRING_PATTERN = re.compile(
    r"""'(?:\\.|[^'\\])*'|"(?:\\.|[^"\\])*" """,
    flags=re.VERBOSE | re.DOTALL
)


def parse_dialogue(value):
    """
    Convert the stored dialogue string into a Python list of utterances.

    Example input:
    "['Hello .' 'How are you ?']"

    Example output:
    ['Hello .', 'How are you ?']
    """
    if pd.isna(value):
        return []

    text = str(value)
    quoted_tokens = QUOTED_STRING_PATTERN.findall(text)

    utterances = []

    for token in quoted_tokens:
        try:
            utterance = ast.literal_eval(token)
        except (SyntaxError, ValueError):
            utterance = token[1:-1]

        utterance = re.sub(r"\s+", " ", str(utterance)).strip()

        if utterance:
            utterances.append(utterance)

    return utterances


def parse_integer_sequence(value):
    """
    Convert strings such as '[0 0 4 4]' into [0, 0, 4, 4].
    """
    if pd.isna(value):
        return []

    return [
        int(number)
        for number in re.findall(r"-?\d+", str(value))
    ]

Test the parser.

In [6]:
sample_dialogue = parse_dialogue(train_raw.loc[0, "dialog"])
sample_acts = parse_integer_sequence(train_raw.loc[0, "act"])
sample_emotions = parse_integer_sequence(
    train_raw.loc[0, "emotion"]
)

print("Utterances:", len(sample_dialogue))
print("Act labels:", len(sample_acts))
print("Emotion labels:", len(sample_emotions))

for index, (utterance, act, emotion) in enumerate(
    zip(sample_dialogue, sample_acts, sample_emotions)
):
    print(
        f"Turn {index:02d} | Act={act} | "
        f"Emotion={emotion} | {utterance}"
    )

Utterances: 10
Act labels: 10
Emotion labels: 10
Turn 00 | Act=3 | Emotion=0 | Say , Jim , how about going for a few beers after dinner ?
Turn 01 | Act=4 | Emotion=0 | You know that is tempting but is really not good for our fitness .
Turn 02 | Act=2 | Emotion=0 | What do you mean ? It will help us to relax .
Turn 03 | Act=2 | Emotion=0 | Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?
Turn 04 | Act=2 | Emotion=0 | I guess you are right.But what shall we do ? I don't feel like sitting at home .
Turn 05 | Act=3 | Emotion=0 | I suggest a walk over to the gym where we can play singsong and meet some of our friends .
Turn 06 | Act=4 | Emotion=4 | That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .
Turn 07 | Act=1 | Emotion=4 | Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .
Turn 08 | Act=3 | Emotion=

Parse every split

In [ ]:
def parse_split(dataframe, split_name):
    parsed = dataframe.copy()

    parsed["dialogue_id"] = [
        f"{split_name}_{index:05d}"
        for index in range(len(parsed))
    ]

    parsed["split"] = split_name
    parsed["dialogue_list"] = parsed["dialog"].apply(
        parse_dialogue
    )

    parsed["act_list"] = parsed["act"].apply(
        parse_integer_sequence
    )

    parsed["emotion_list"] = parsed["emotion"].apply(
        parse_integer_sequence
    )

    return parsed


train_dialogues = parse_split(train_raw, "train")
validation_dialogues = parse_split(
    validation_raw,
    "validation"
)
test_dialogues = parse_split(test_raw, "test")

Validate sequence alignment

In [8]:
def validate_dialogues(dataframe, split_name):
    problem_rows = []

    for row in dataframe.itertuples():
        number_of_utterances = len(row.dialogue_list)
        number_of_acts = len(row.act_list)
        number_of_emotions = len(row.emotion_list)

        if not (
            number_of_utterances
            == number_of_acts
            == number_of_emotions
        ):
            problem_rows.append({
                "dialogue_id": row.dialogue_id,
                "utterances": number_of_utterances,
                "acts": number_of_acts,
                "emotions": number_of_emotions
            })

    problems = pd.DataFrame(problem_rows)

    print(
        f"{split_name}: "
        f"{len(dataframe)} dialogues, "
        f"{len(problems)} alignment problems"
    )

    return problems


train_problems = validate_dialogues(
    train_dialogues,
    "train"
)

validation_problems = validate_dialogues(
    validation_dialogues,
    "validation"
)

test_problems = validate_dialogues(
    test_dialogues,
    "test"
)

train: 11118 dialogues, 0 alignment problems
validation: 1000 dialogues, 0 alignment problems
test: 1000 dialogues, 0 alignment problems


Flatten to utterance-level data

In [9]:
def flatten_dialogues(dialogue_dataframe):
    utterance_rows = []

    for dialogue in dialogue_dataframe.itertuples():
        utterances = dialogue.dialogue_list
        acts = dialogue.act_list
        emotions = dialogue.emotion_list

        for turn_index, utterance in enumerate(utterances):
            utterance_rows.append({
                "split": dialogue.split,
                "dialogue_id": dialogue.dialogue_id,
                "turn_index": turn_index,

                # DailyDialog conversations alternate speakers.
                "speaker_id": turn_index % 2,

                "utterance": utterance,
                "act_label": acts[turn_index],
                "emotion_label": emotions[turn_index],

                "dialogue_length": len(utterances),
                "utterance_length_words": len(
                    utterance.split()
                ),
                "utterance_length_characters": len(
                    utterance
                )
            })

    return pd.DataFrame(utterance_rows)


train_df = flatten_dialogues(train_dialogues)
validation_df = flatten_dialogues(validation_dialogues)
test_df = flatten_dialogues(test_dialogues)

print("Train utterances:", len(train_df))
print("Validation utterances:", len(validation_df))
print("Test utterances:", len(test_df))

Train utterances: 87170
Validation utterances: 8069
Test utterances: 7740


Map numerical levels 

In [10]:
EMOTION_LABELS = {
    0: "neutral",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise"
}

ACT_LABELS = {
    1: "inform",
    2: "question",
    3: "directive",
    4: "commissive"
}


def add_readable_labels(dataframe):
    result = dataframe.copy()

    result["emotion_name"] = result[
        "emotion_label"
    ].map(EMOTION_LABELS)

    result["act_name"] = result[
        "act_label"
    ].map(ACT_LABELS)

    if result["emotion_name"].isna().any():
        unknown = result.loc[
            result["emotion_name"].isna(),
            "emotion_label"
        ].unique()

        raise ValueError(
            f"Unknown emotion labels found: {unknown}"
        )

    if result["act_name"].isna().any():
        unknown = result.loc[
            result["act_name"].isna(),
            "act_label"
        ].unique()

        raise ValueError(
            f"Unknown dialogue act labels found: {unknown}"
        )

    return result


train_df = add_readable_labels(train_df)
validation_df = add_readable_labels(validation_df)
test_df = add_readable_labels(test_df)

Build speaker aware-contex
This version is more informative than simply joining previous utterances with [SEP].

In [11]:
def build_context_features(dataframe):
    dataframe = dataframe.sort_values(
        ["dialogue_id", "turn_index"]
    ).reset_index(drop=True).copy()

    output_groups = []

    for _, group in dataframe.groupby(
        "dialogue_id",
        sort=False
    ):
        group = group.sort_values("turn_index").copy()

        utterances = group["utterance"].tolist()
        speakers = group["speaker_id"].tolist()

        previous_1 = []
        previous_3 = []
        full_previous = []

        for current_index in range(len(group)):
            formatted_previous_turns = []

            for previous_index in range(current_index):
                speaker_token = (
                    "[SPEAKER_A]"
                    if speakers[previous_index] == 0
                    else "[SPEAKER_B]"
                )

                formatted_previous_turns.append(
                    f"{speaker_token} "
                    f"{utterances[previous_index]}"
                )

            previous_1.append(
                " [SEP] ".join(
                    formatted_previous_turns[-1:]
                )
            )

            previous_3.append(
                " [SEP] ".join(
                    formatted_previous_turns[-3:]
                )
            )

            full_previous.append(
                " [SEP] ".join(
                    formatted_previous_turns
                )
            )

        current_speaker_tokens = [
            (
                "[SPEAKER_A]"
                if speaker_id == 0
                else "[SPEAKER_B]"
            )
            for speaker_id in speakers
        ]

        group["previous_1"] = previous_1
        group["previous_3"] = previous_3
        group["full_previous_context"] = full_previous

        group["text_no_context"] = [
            f"{speaker} {utterance}"
            for speaker, utterance in zip(
                current_speaker_tokens,
                utterances
            )
        ]

        group["text_context_1"] = [
            (
                f"{context} [CURRENT] "
                f"{speaker} {utterance}"
                if context
                else f"[CURRENT] {speaker} {utterance}"
            )
            for context, speaker, utterance in zip(
                previous_1,
                current_speaker_tokens,
                utterances
            )
        ]

        group["text_context_3"] = [
            (
                f"{context} [CURRENT] "
                f"{speaker} {utterance}"
                if context
                else f"[CURRENT] {speaker} {utterance}"
            )
            for context, speaker, utterance in zip(
                previous_3,
                current_speaker_tokens,
                utterances
            )
        ]

        group["text_full_context"] = [
            (
                f"{context} [CURRENT] "
                f"{speaker} {utterance}"
                if context
                else f"[CURRENT] {speaker} {utterance}"
            )
            for context, speaker, utterance in zip(
                full_previous,
                current_speaker_tokens,
                utterances
            )
        ]

        output_groups.append(group)

    return pd.concat(
        output_groups,
        ignore_index=True
    )


train_df = build_context_features(train_df)
validation_df = build_context_features(validation_df)
test_df = build_context_features(test_df)

inspecting the results 

In [12]:
display(
    train_df[
        [
            "dialogue_id",
            "turn_index",
            "text_no_context",
            "text_context_1",
            "text_context_3",
            "emotion_name"
        ]
    ].head(15)
)

,dialogue_id,turn_index,text_no_context,text_context_1,text_context_3,emotion_name
0,train_00000,0,"[SPEAKER_A] Say , Jim , how about going for a ...","[CURRENT] [SPEAKER_A] Say , Jim , how about go...","[CURRENT] [SPEAKER_A] Say , Jim , how about go...",neutral
1,train_00000,1,[SPEAKER_B] You know that is tempting but is r...,"[SPEAKER_A] Say , Jim , how about going for a ...","[SPEAKER_A] Say , Jim , how about going for a ...",neutral
2,train_00000,2,[SPEAKER_A] What do you mean ? It will help us...,[SPEAKER_B] You know that is tempting but is r...,"[SPEAKER_A] Say , Jim , how about going for a ...",neutral
3,train_00000,3,[SPEAKER_B] Do you really think so ? I don't ....,[SPEAKER_A] What do you mean ? It will help us...,"[SPEAKER_A] Say , Jim , how about going for a ...",neutral
4,train_00000,4,[SPEAKER_A] I guess you are right.But what sha...,[SPEAKER_B] Do you really think so ? I don't ....,[SPEAKER_B] You know that is tempting but is r...,neutral
5,train_00000,5,[SPEAKER_B] I suggest a walk over to the gym w...,[SPEAKER_A] I guess you are right.But what sha...,[SPEAKER_A] What do you mean ? It will help us...,neutral
6,train_00000,6,[SPEAKER_A] That's a good idea . I hear Mary a...,[SPEAKER_B] I suggest a walk over to the gym w...,[SPEAKER_B] Do you really think so ? I don't ....,happiness
7,train_00000,7,[SPEAKER_B] Sounds great to me ! If they are w...,[SPEAKER_A] That's a good idea . I hear Mary a...,[SPEAKER_A] I guess you are right.But what sha...,happiness
8,train_00000,8,[SPEAKER_A] Good.Let ' s go now .,[SPEAKER_B] Sounds great to me ! If they are w...,[SPEAKER_B] I suggest a walk over to the gym w...,happiness
9,train_00000,9,[SPEAKER_B] All right .,[SPEAKER_A] Good.Let ' s go now . [CURRENT] [S...,[SPEAKER_A] That's a good idea . I hear Mary a...,happiness


Create shuffled-context and unrelated-context controls
These are important because they test whether improvements come from meaningful context rather than simply longer text.

In [13]:
def add_shuffled_context(dataframe, seed=42):
    rng = np.random.default_rng(seed)
    result = dataframe.copy()

    shuffled_texts = []

    for _, group in result.groupby(
        "dialogue_id",
        sort=False
    ):
        group = group.sort_values("turn_index")

        for row in group.itertuples():
            if not row.previous_3:
                shuffled_texts.append(
                    row.text_no_context
                )
                continue

            turns = row.previous_3.split(" [SEP] ")
            rng.shuffle(turns)

            shuffled_context = " [SEP] ".join(turns)

            shuffled_texts.append(
                f"{shuffled_context} "
                f"[CURRENT] {row.text_no_context}"
            )

    result["text_shuffled_context"] = shuffled_texts

    return result


train_df = add_shuffled_context(train_df, seed=SEED)
validation_df = add_shuffled_context(
    validation_df,
    seed=SEED
)
test_df = add_shuffled_context(test_df, seed=SEED)

create unlearated context from another dialog.

In [14]:
def add_unrelated_context(dataframe, seed=42):
    rng = np.random.default_rng(seed)
    result = dataframe.copy()

    candidate_contexts = (
        result.loc[
            result["previous_3"].str.len() > 0,
            "previous_3"
        ]
        .drop_duplicates()
        .tolist()
    )

    unrelated_texts = []

    for row in result.itertuples():
        if not candidate_contexts:
            unrelated_texts.append(
                row.text_no_context
            )
            continue

        random_context = rng.choice(
            candidate_contexts
        )

        unrelated_texts.append(
            f"{random_context} "
            f"[CURRENT] {row.text_no_context}"
        )

    result["text_unrelated_context"] = unrelated_texts

    return result


train_df = add_unrelated_context(train_df, seed=SEED)
validation_df = add_unrelated_context(
    validation_df,
    seed=SEED
)
test_df = add_unrelated_context(test_df, seed=SEED)

Check split leakage

In [15]:
train_ids = set(train_df["dialogue_id"])
validation_ids = set(validation_df["dialogue_id"])
test_ids = set(test_df["dialogue_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

print("No dialogue IDs overlap across splits.")

No dialogue IDs overlap across splits.


check exact utterance overlap

In [16]:
def normalized_text_set(dataframe):
    return set(
        dataframe["utterance"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


train_texts = normalized_text_set(train_df)
validation_texts = normalized_text_set(
    validation_df
)
test_texts = normalized_text_set(test_df)

print(
    "Train-validation repeated utterances:",
    len(train_texts & validation_texts)
)

print(
    "Train-test repeated utterances:",
    len(train_texts & test_texts)
)

print(
    "Validation-test repeated utterances:",
    len(validation_texts & test_texts)
)

Train-validation repeated utterances: 1859
Train-test repeated utterances: 2106
Validation-test repeated utterances: 303


save processed dataset

In [17]:
train_df.to_csv(
    PROCESSED_DIR / "train_utterances.csv",
    index=False
)

validation_df.to_csv(
    PROCESSED_DIR / "validation_utterances.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test_utterances.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
